In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

cwd = os.getcwd()
root = cwd.split("BernsteinMartingaleNet")[0] + "BernsteinMartingaleNet"
if root not in sys.path:
    sys.path.append(root)

from lib.utils import get_sequence_data, train_model, get_sequence_data_by_month
from lib.BLogistic import BLogistic
from lib.DistHead import NormalHead, StudentTHead, SkewedStudentTHead
from lib.Michenkow import Michenkow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


folder_path = root + r"/MarketData/historical_data"
context_window = 60
X, Y     = get_sequence_data(folder_path, context_window)
dof      = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 40000
test_size = 40000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
test_indices = indices[dev_size:dev_size + test_size]
train_indices = indices[dev_size + test_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]
test_X = simple_X[test_indices, :]
test_Y = simple_Y[test_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
test_X = test_X / std
test_Y = test_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

using cached data from /home/jkim/coding_projects/BernsteinMartingaleNet/MarketData/historical_data/spy_1min_data_context_60.npz
train_X torch.Size([332426, 60]) train_Y torch.Size([332426, 1]) dev_X torch.Size([40000, 60]) dev_Y torch.Size([40000, 1])
std tensor(0.0004, device='cuda:0') tensor(1.0263, device='cuda:0')


In [2]:

train_xs, train_ys, dev_xs, dev_ys, test_xs, test_ys = get_sequence_data_by_month(folder_path, context_window, 8, 8, True)
print(train_xs.shape, train_ys.shape, dev_xs.shape, dev_ys.shape, test_xs.shape, test_ys.shape)

skipping 2020-11-27
skipping 2024-07-03
skipping 2024-11-29
skipping 2020-12-24
skipping 2023-11-24
skipping 2023-07-03
skipping 2022-11-25
skipping 2025-07-03
skipping 2024-12-24
skipping 2021-11-26
train months ['2024-09' '2025-08' '2023-06' '2024-04' '2025-02' '2023-05' '2022-10'
 '2025-01' '2020-11' '2024-11' '2021-10' '2022-08' '2023-08' '2024-02'
 '2022-04' '2025-07' '2022-12' '2021-09' '2021-01' '2022-05' '2022-03'
 '2021-11' '2023-01' '2024-06' '2023-12' '2021-03' '2023-04' '2021-12'
 '2021-02' '2025-09' '2024-01' '2022-01' '2023-10' '2020-10' '2023-07'
 '2024-10' '2024-08' '2020-12' '2023-03' '2021-05' '2025-06' '2025-03'
 '2023-09' '2022-07' '2021-06']
dev months ['2025-04' '2024-07' '2024-12' '2024-03' '2022-06' '2021-07' '2023-02'
 '2024-05']
test months ['2021-08' '2023-11' '2025-05' '2021-04' '2022-09' '2025-10' '2022-02'
 '2022-11']
num months 45 8 8
(309154, 60, 3) (309154,) (54615, 60, 3) (54615,) (48657, 60, 3) (48657,)


In [ ]:
np.std(train_ys)

17404804.304132264

In [ ]:
lr = 0.002
decay_step = 50
decay_gamma = 0.5
weight_decay = 0
num_steps = 1
batch_size = 512 * 8

In [ ]:
# with lr = 0.002
model = Michenkow(context_window, NormalHead(), device)
train_path = "Train_Normal_test/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
model = LSTMProbNNDist(context_window, NormalHead(), device)
model.load_state_dict(torch.load("Train_Normal/model_80.pth"))#load
train_path = "Train_Normal_Warm/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, 1e-4, weight_decay, 100, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
model = LSTMProbNNDist(context_window, NormalHead(), device)
model.load_state_dict(torch.load("Train_Normal_Warm/model_90.pth"))#load
train_path = "Train_Normal_Warmer/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, 1e-4, weight_decay, 140, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
# with lr = 0.01
model = LSTMProbNNDist(context_window, NormalHead(), device)
train_path = "Train_Normal/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect() 

In [ ]:
model = LSTMProbNNDist(context_window, StudentTHead(), device)
train_path = "Train_StudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size,
    device=device, output_folder=train_path, lr_decay_step=decay_step, lr_decay_gamma=decay_gamma)

In [ ]:
weight_decay = 0.002
model = LSTMProbNNDist(context_window, StudentTHead(), device)
train_path = "Train_StudentT_Regularized/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size,
    device=device, output_folder=train_path, lr_decay_step=decay_step, lr_decay_gamma=decay_gamma)


In [ ]:
model = LSTMProbNNDist(context_window, SkewedStudentTHead(), device)
train_path = "Train_SkewedStudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
dof = 16
model = LSTMProbNNDist(context_window, BLogistic(dof - 2, device), device)
train_path = "Train_BLogistic/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)